In [1]:
import os

CPU_THREADS = 16
CPU_ONLY = True

if CPU_ONLY:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)

# Train agent

In [2]:
from setup import setup_modules

setup_modules()

In [3]:
from building_maintenance_agents.envs import (
  			BuildingIncidentEnv,
     		AgentAction, 
  			AgentActionType,
     		IncidentObservation,
            CurrentReward,
            RewardConfig,
)
      
from building_maintenance_agents.envs.training import (
  			make_env, 
     		train_agent,
  			evaluate_agent, 
     		compare_strategies
)


## Test env

In [4]:
env = BuildingIncidentEnv(
		house_type="16",
		max_steps=100,
		render_mode="human",
		incident_probability=0.05
)

obs, info = env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")

2026-05-12 23:29:06.757 | INFO     | building_maintenance_agents.incident_simulator.incident_simulator:__init__:58 - IncidentSimulator initialized with 513 nodes, 1372 edges
2026-05-12 23:29:06.758 | INFO     | building_maintenance_agents.incident_simulator.incident_simulator:__init__:59 - Base incident probability: 0.05
2026-05-12 23:29:06.809 | INFO     | building_maintenance_agents.incident_simulator.incident_simulator:__init__:58 - IncidentSimulator initialized with 513 nodes, 1372 edges
2026-05-12 23:29:06.811 | INFO     | building_maintenance_agents.incident_simulator.incident_simulator:__init__:59 - Base incident probability: 0.05


Observation shape: (18537,)
Action space: Box([0.  0.  0.  0.5], [6.000e+00 1.371e+03 1.000e+00 2.000e+00], (4,), float32)


In [5]:
total_reward = 0
for step in range(20):
		action = env.action_space.sample()
		obs, reward, terminated, truncated, info = env.step(action)
		total_reward += reward
		
		if terminated or truncated:
				break

print(f"Episode finished with reward: {total_reward:.4f}")


Step: 1
Resources: 100.0/100.0
Active incidents: 0
Total reward: 1.10

Statistics: total incidents=0, max active=0

Step: 2
Resources: 97.0/100.0
Active incidents: 0
Total reward: 1.20

Statistics: total incidents=0, max active=0

Step: 3
Resources: 97.5/100.0
Active incidents: 0
Total reward: 2.80

Statistics: total incidents=0, max active=0

Step: 4
Resources: 97.7/100.0
Active incidents: 0
Total reward: 4.10

Statistics: total incidents=0, max active=0

Step: 5
Resources: 95.4/100.0
Active incidents: 0
Total reward: 3.20

Statistics: total incidents=0, max active=0

Step: 6
Resources: 95.6/100.0
Active incidents: 0
Total reward: 3.30

Statistics: total incidents=0, max active=0

Step: 7
Resources: 95.8/100.0
Active incidents: 0
Total reward: 3.90

Statistics: total incidents=0, max active=0

Step: 8
Resources: 94.5/100.0
Active incidents: 0
Total reward: 4.00

Statistics: total incidents=0, max active=0

Step: 9
Resources: 94.3/100.0
Active incidents: 0
Total reward: 4.10

Statisti

In [6]:
# compare_strategies()

In [7]:
from pathlib import Path
from datetime import datetime
import csv
import numpy as np
from loguru import logger


def silence_project_logging():
    logger.disable("building_maintenance_agents")


def configure_notebook_project_logging(log_path, file_level="INFO"):
    logger.enable("building_maintenance_agents")
    logger.remove()
    logger.add(log_path, level=file_level, enqueue=True)


def write_csv(path, fieldnames, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def evaluate_agent_with_trace(
    model_path,
    env_config,
    n_episodes=5,
    summary_csv_path=None,
    steps_csv_path=None,
    events_csv_path=None,
):
    from stable_baselines3 import PPO, A2C

    model = PPO.load(model_path) if "PPO" in model_path else A2C.load(model_path)
    env = BuildingIncidentEnv(**env_config)

    episode_rewards = []
    episode_lengths = []
    episode_summaries = []
    step_rows = []
    event_rows = []

    for episode in range(n_episodes):
        obs, info = env.reset()
        initial_stats = env.get_metrics()["incident_stats"]
        initial_incidents = int(initial_stats["total_incidents"])
        if initial_incidents > 0:
            event_rows.append({
                "episode": episode + 1,
                "step": 0,
                "event": "initial_incidents",
                "count": initial_incidents,
                "action": "RESET",
                "reward": 0.0,
                "total_reward": 0.0,
                "active_incidents": int(initial_stats["active_incidents"]),
                "new_incidents": initial_incidents,
                "resolved_incidents": 0,
                "resources": float(env.resources),
            })

        done = False
        episode_reward = 0.0
        step = 0
        step_new_incidents = 0

        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            step += 1

            row = {
                "episode": episode + 1,
                "step": step,
                "action": info.get("action"),
                "reward": float(reward),
                "total_reward": float(info.get("total_reward", episode_reward)),
                "active_incidents": int(info.get("active_incidents", 0)),
                "new_incidents": int(info.get("new_incidents", 0)),
                "resolved_incidents": int(info.get("resolved_incidents", 0)),
                "resources": float(info.get("resources", 0.0)),
            }
            step_rows.append(row)

            step_new_incidents += row["new_incidents"]
            if row["new_incidents"] > 0:
                event_rows.append({**row, "event": "new_incidents", "count": row["new_incidents"]})
            if row["resolved_incidents"] > 0:
                event_rows.append({**row, "event": "resolved_incidents", "count": row["resolved_incidents"]})

            done = terminated or truncated

        episode_rewards.append(episode_reward)
        episode_lengths.append(step)

        metrics = env.get_metrics()
        incident_stats = metrics["incident_stats"]
        episode_summary = {
            "episode": episode + 1,
            "reward": float(episode_reward),
            "steps": int(step),
            "initial_incidents": initial_incidents,
            "step_new_incidents": int(step_new_incidents),
            "total_incidents": int(incident_stats["total_incidents"]),
            "max_active_incidents": int(incident_stats["max_active_incidents"]),
            "active_incidents_at_end": int(incident_stats["active_incidents"]),
            "resources_used": float(metrics["resources_used"]),
            "resources_left": float(metrics["resources_left"]),
        }
        episode_summaries.append(episode_summary)


    total_eval_incidents = sum(summary["total_incidents"] for summary in episode_summaries)
    total_step_new_incidents = sum(summary["step_new_incidents"] for summary in episode_summaries)
    total_initial_incidents = sum(summary["initial_incidents"] for summary in episode_summaries)
    logger.info(
        "Evaluation incidents: total={}, initial={}, new_during_steps={}, mean_per_episode={:.2f}",
        total_eval_incidents,
        total_initial_incidents,
        total_step_new_incidents,
        np.mean([s["total_incidents"] for s in episode_summaries]),
    )

    if summary_csv_path is not None:
        write_csv(summary_csv_path, list(episode_summaries[0].keys()), episode_summaries)
    if steps_csv_path is not None:
        write_csv(steps_csv_path, list(step_rows[0].keys()), step_rows)
    if events_csv_path is not None:
        event_fields = ["episode", "step", "event", "count", "action", "reward", "total_reward", "active_incidents", "new_incidents", "resolved_incidents", "resources"]
        write_csv(events_csv_path, event_fields, event_rows)

    env.close()
    return episode_rewards, episode_lengths, episode_summaries


def log_experiment_result(
    csv_path,
    env_config,
    algorithm,
    total_timesteps,
    n_envs,
    episode_rewards,
    episode_lengths,
    episode_summaries,
    model_path,
    reward_variant,
    experiment_name,
):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    episode_incidents = [summary["total_incidents"] for summary in episode_summaries]
    episode_active_end = [summary["active_incidents_at_end"] for summary in episode_summaries]
    episode_resources_used = [summary["resources_used"] for summary in episode_summaries]

    row = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "algorithm": algorithm,
        "reward_variant": reward_variant,
        "experiment_name": experiment_name,
        "total_timesteps": total_timesteps,
        "n_envs": n_envs,
        "house_type": env_config.get("house_type"),
        "max_steps": env_config.get("max_steps"),
        "incident_probability": env_config.get("incident_probability"),
        "resource_budget": env_config.get("resource_budget"),
        "enable_spread": env_config.get("enable_spread"),
        "mean_reward": float(np.mean(episode_rewards)),
        "std_reward": float(np.std(episode_rewards)),
        "min_reward": float(np.min(episode_rewards)),
        "max_reward": float(np.max(episode_rewards)),
        "mean_steps": float(np.mean(episode_lengths)),
        "std_steps": float(np.std(episode_lengths)),
        "mean_incidents": float(np.mean(episode_incidents)),
        "std_incidents": float(np.std(episode_incidents)),
        "min_incidents": int(np.min(episode_incidents)),
        "max_incidents": int(np.max(episode_incidents)),
        "mean_active_end": float(np.mean(episode_active_end)),
        "mean_resources_used": float(np.mean(episode_resources_used)),
        "model_path": model_path,
    }

    fieldnames = list(row.keys())
    write_header = not csv_path.exists() or csv_path.stat().st_size == 0
    if not write_header:
        with csv_path.open(newline="") as file:
            existing_header = next(csv.reader(file), [])
        write_header = existing_header != fieldnames

    with csv_path.open("a", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    return row


## Train PPO

In [ ]:
base_env_config = {
		"house_type": "16",
		"max_steps": 100,
		"incident_probability": 1.0, #?
		"resource_budget": 100.0,
		"enable_spread": True
}

baseline_config = {
    **base_env_config,
}

incident_focused_config = {
    **base_env_config,
    "reward_config": RewardConfig(
        active_incident_penalty_multiplier=0.5,
        resource_efficiency_reward=0.05,
    ),
}

resource_efficient_config = {
    **base_env_config,
    "reward_config": RewardConfig(
        resource_efficiency_reward=0.3,
        deploy_resolved_bonus=12,
        repair_resolved_bonus=9,
        call_backup_reward=-2,
    ),
}

test_config = {
    **base_env_config,
    "incident_probability": 1.0,
    "reward_config": RewardConfig(
        insufficient_resources_penalty=-10.0,
        invalid_target_penalty=-8.0,

        no_incidents_bonus=0.0,
        resource_efficiency_reward=0.0,

        monitor_no_incidents_reward=3.0,
        monitor_active_incidents_penalty=-10.0,

        withdraw_no_incidents_reward=-1.0,
        withdraw_active_incidents_penalty=-4.0,

        repair_no_incidents_penalty=-16.0,
        deploy_no_incidents_penalty=-16.0,

        deploy_severity_multiplier=0.0,
        deploy_resolved_bonus=0.0,

        repair_effectiveness_multiplier=1.0,
        repair_severity_multiplier=35.0,
        repair_resolved_bonus=90.0,

        shut_off_water_reward=1.0,
        shut_off_water_penalty=-8.0,

        inspect_valid_target_reward=-0.5,
        inspect_invalid_target_penalty=-8.0,

        backup_amount_multiplier=0.0,
        call_backup_reward=-12.0,

        active_incident_penalty_multiplier=3.0,
        too_many_incidents_penalty=-20.0,

        max_incident_age=25,
        min_resolve_bonus_factor=0.2,
    ),
}

balanced_config = {
    **base_env_config,
    "reward_config": RewardConfig(
        active_incident_penalty_multiplier=0.3,
        deploy_resolved_bonus=25,
        repair_resolved_bonus=20,
    ),
}

reward_experiment_configs = {
    "baseline": baseline_config,
    "incident_focused": incident_focused_config,
    "resource_efficient": resource_efficient_config,
    "test": test_config,
    "balanced": balanced_config,
}


In [9]:

reward_variant = "test"
env_config = reward_experiment_configs[reward_variant]

algorithm = "PPO"
total_timesteps = 20000 # было 100000
n_envs = 4
show_progress_bar = True
experiment_name = f"{algorithm}_{reward_variant}"
experiment_dir = f"./models/{experiment_name}"
model_path = f"{experiment_dir}/{algorithm}_incident_model.zip"
csv_log_path = f"{experiment_dir}/experiment_results.csv"
project_log_file = f"{experiment_dir}/project.log"
summary_csv_path = f"{experiment_dir}/evaluation_summary.csv"
steps_csv_path = f"{experiment_dir}/evaluation_steps.csv"
events_csv_path = f"{experiment_dir}/evaluation_incident_events.csv"

silence_project_logging()

model = train_agent(
		env_config,
		total_timesteps=total_timesteps,
		algorithm=algorithm,
		n_envs=n_envs,
        save_path=experiment_dir,
        progress_bar=show_progress_bar,
)

configure_notebook_project_logging(
    project_log_file,
    file_level="INFO",
)

episode_rewards, episode_lengths, episode_summaries = evaluate_agent_with_trace(
    model_path,
    env_config=env_config,
    n_episodes=5,
    summary_csv_path=summary_csv_path,
    steps_csv_path=steps_csv_path,
    events_csv_path=events_csv_path,
)
silence_project_logging()

experiment_log = log_experiment_result(
    csv_path=csv_log_path,
    env_config=env_config,
    algorithm=algorithm,
    total_timesteps=total_timesteps,
    n_envs=n_envs,
    episode_rewards=episode_rewards,
    episode_lengths=episode_lengths,
    episode_summaries=episode_summaries,
    model_path=model_path,
    reward_variant=reward_variant,
    experiment_name=experiment_name,
)


Using cpu device
Logging to ./models/PPO_test/tensorboard/PPO_incident_control_11


/home/fedor/Projects/building_maintenance_agents/.venv/lib/python3.12/site-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x7bd8c65a6c30> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7bd8c530b830>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


Training:   0%|          | 0/20000 [00:00<?, ?it/s]

-----------------------------
| time/              |      |
|    fps             | 160  |
|    iterations      | 1    |
|    time_elapsed    | 51   |
|    total_timesteps | 8192 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 120          |
|    iterations           | 2            |
|    time_elapsed         | 135          |
|    total_timesteps      | 16384        |
| train/                  |              |
|    approx_kl            | 0.0041718087 |
|    clip_fraction        | 0.0353       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.69        |
|    explained_variance   | -1.43e-05    |
|    learning_rate        | 0.0003       |
|    loss                 | 2.12e+04     |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00194     |
|    std                  | 1            |
|    value_loss           | 4.76e+04     |
----------------

In [10]:
experiment_log

{'timestamp': '2026-05-12T23:33:50',
 'algorithm': 'PPO',
 'reward_variant': 'test',
 'experiment_name': 'PPO_test',
 'total_timesteps': 20000,
 'n_envs': 4,
 'house_type': '16',
 'max_steps': 100,
 'incident_probability': 2.0,
 'resource_budget': 100.0,
 'enable_spread': True,
 'mean_reward': -1664.1795139312676,
 'std_reward': 42.89694294450547,
 'min_reward': -1730.8569113663086,
 'max_reward': -1613.1453189759054,
 'mean_steps': 100.0,
 'std_steps': 0.0,
 'mean_incidents': 2.0,
 'std_incidents': 0.6324555320336759,
 'min_incidents': 1,
 'max_incidents': 3,
 'mean_active_end': 1.2,
 'mean_resources_used': 125.0,
 'model_path': './models/PPO_test/PPO_incident_model.zip'}